<a href="https://colab.research.google.com/github/ghinaiyariken/Fly_Rank_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghinaiyariken/Fly_Rank_Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Ranking / scoring.** My lane is Refresh / Content Opportunity Scoring, so the output should be an ordered review queue rather than a yes/no prediction. The decision is which pages an SEO/content team should inspect first when review capacity is limited. A ranking is a better fit than classification because several pages can all deserve attention, but the team still needs to know which ones to look at first. The eventual output should be a priority score plus reason codes, not an automatic instruction to edit a page.


In [4]:
# Load the starter data and verify the lane's basic setup.
import pandas as pd

DATA_PATH = "content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

print(f"Starter rows: {len(df):,}")
print(f"Unique clients: {df['client_id'].nunique():,}")
print("Unit of analysis: one row per pseudonymized content item.")


Starter rows: 30,000
Unique clients: 32
Unit of analysis: one row per pseudonymized content item.


## 2. Target or proxy

The starter dataset does not contain a clean **future-window outcome** for whether a page later became a useful refresh candidate. Therefore, for this framing exercise I will use a **transparent review-opportunity proxy**, not a ground-truth ML label.

The provisional proxy gives one point for each of these observed conditions: (1) at least 100 impressions in 90 days and a current `down` trend, (2) average position from 1–10 with content age at least 180 days, and (3) at least 500 impressions, average position from 1–20, and CTR below 0.5%. The proxy ranges from 0 to 3 and is only a way to make the ranking idea concrete on the starter data.

I will **not** claim that this proxy is the truth, and I will not train a final model simply to reproduce it. For the later ML stage, I should build a future-window observed outcome from the warehouse data and keep the feature window strictly before that outcome window.


In [5]:
# Build the transparent provisional proxy described above.
visible_down = (
    (df["impressions_90d"] >= 100)
    & (df["trend_direction"] == "down")
)
page_one_old = (
    (df["avg_position"] > 0)
    & (df["avg_position"] <= 10)
    & (df["content_age_days"] >= 180)
)
low_ctr_visible = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

df["review_proxy_score"] = (
    visible_down.astype(int)
    + page_one_old.astype(int)
    + low_ctr_visible.astype(int)
)

print("Proxy score distribution:")
print(df["review_proxy_score"].value_counts().sort_index().to_string())
print(f"Pages with proxy score >= 2: {(df['review_proxy_score'] >= 2).sum():,}")


Proxy score distribution:
review_proxy_score
0    10377
1    11216
2     6450
3     1957
Pages with proxy score >= 2: 8,407


## 3. Success metric

**Primary metric: Precision@20.** The real workflow is a ranked review queue, so I care about how many of the first 20 recommended pages are later confirmed as useful review candidates. My provisional policy target is **at least 12 of the top 20 (60%)** being confirmed useful, while also beating the transparent baseline.

The 60% figure is a project policy choice, not a universal benchmark. I will only evaluate it once I have a defensible future-window outcome label. If the team's actual review capacity is different, K should be changed to match that capacity.


In [6]:
# Check the metric definition and show the current top-20 proxy ranking.
K = 20
top20 = df.sort_values(
    ["review_proxy_score", "impressions_90d"],
    ascending=[False, False]
).head(K)

print(f"Evaluation metric: Precision@{K}")
print("Provisional policy target: >= 12/20 = 60% confirmed useful candidates.")
print(f"Top-{K} rows available for later review: {len(top20)}")


Evaluation metric: Precision@20
Provisional policy target: >= 12/20 = 60% confirmed useful candidates.
Top-20 rows available for later review: 20


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item/page**, with trailing-90-day aggregate search, traffic, freshness, and engagement signals in the starter dataset. The model output would therefore rank individual content items for human review.

The starter CSV is appropriate for this initial framing because it contains one row per content item and the observable signals needed to demonstrate the lane. The later warehouse version can add time-aware history so that the final target can be measured in a genuinely later window.


In [7]:
# Show the actual dataframe slice representing the unit of analysis.
slice_cols = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "trend_direction",
    "review_proxy_score",
]

lane_slice = df[slice_cols].copy()

print("One row represents one pseudonymized content item.")
print(f"Rows in lane slice: {len(lane_slice):,}")
display(lane_slice.head(10))


One row represents one pseudonymized content item.
Rows in lane slice: 30,000


,content_id,impressions_90d,clicks_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,trend_direction,review_proxy_score
0,content_304f48230142,3803,29,17,10.6,0.76,187,20,down,1
1,content_a1fb4e703a9e,15320,7,9,20.3,0.05,445,25,down,1
2,content_9aa793d4d895,12581,11,11,36.5,0.09,141,20,down,1
3,content_331d6c4de07b,11751,58,78,6.2,0.49,463,22,stable,2
4,content_d99b7a2d90ca,19140,24,145,44.0,0.13,263,14,down,1
5,content_d4084a4bc775,3970,1,5,8.5,0.03,147,20,down,2
6,content_9a34b442b552,20,0,1,7.0,0.00,90,20,down,0
7,content_a63219c6e95a,1724,1,28,21.2,0.06,445,22,stable,0
8,content_5e6c160719bc,32574,29,68,46.0,0.09,90,20,down,1
9,content_c27558df2b0c,1240,2,3,4.9,0.16,257,104,down,3


## 5. Why ML beats a fixed rule here

A fixed rule is the right baseline and may turn out to be good enough. ML only earns its place if it can improve the ranked queue beyond that baseline.

The pattern is potentially messy because review context can involve several interacting signals at once: search visibility, average position, CTR, traffic, content age, freshness, and recent movement. A simple rule has to choose hard cutoffs for each signal and may miss combinations such as a highly visible older page with a modest CTR problem versus a low-visibility page with a large recent drop.

So the plan is **baseline first, ML second**: build a transparent score that can be explained, then test whether a model produces a more useful top-20 ranking on a future observed outcome. If ML does not beat the simple rule, the rule may be the better solution.


In [8]:
# Check that the proxy uses multiple observable signals rather than one threshold.
print("Proxy components:")
print(f"- Visible + down: {visible_down.sum():,}")
print(f"- Page-one + age >= 180 days: {page_one_old.sum():,}")
print(f"- Visible + position <= 20 + CTR < 0.5%: {low_ctr_visible.sum():,}")

assert df["review_proxy_score"].between(0, 3).all()
assert "trend_direction" in df.columns
print("\nBaseline framing check: proxy combines multiple observed signals.")
print("Important: trend_direction is used only for the provisional proxy; it must not be used as a future-model feature.")


Proxy components:
- Visible + down: 13,152
- Page-one + age >= 180 days: 7,076
- Visible + position <= 20 + CTR < 0.5%: 9,759

Baseline framing check: proxy combines multiple observed signals.
Important: trend_direction is used only for the provisional proxy; it must not be used as a future-model feature.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook uses the starter CSV and shows the actual dataframe grain
- [x] The task type is explicitly ranking/scoring
- [x] The target/proxy is explicitly labeled as a provisional, rule-defined proxy
- [x] Precision@20 is named as the primary future evaluation metric
- [x] No client names, URLs, or private queries are included
- [x] Claims use careful words: observed, provisional, baseline, decision-support
- [ ] The notebook runs top to bottom with no errors in Colab (Runtime → Run all)
- [ ] Committed to my repo under `work/notebooks/` — then submit the repo URL on the card
